## CS 598 PSL Fall 2026
### Project 1 Support Vector Machines and a Comparative Study of Classifiers

Due Date: Monday, November 9 (11.59pm)

- Anirudh Eswara
- Ryan Lenea
- Mark Synowiec
- Robert Walker

This is a project, not a coding assignment. You will not be told which seed to use, which
two variables to plot, or which kernel and grid to try. You will be expected to make those
choices, document them, and defend them with the ideas from class: margin and C, kernels,
hinge versus likelihood, discriminant analysis versus a probability model, and what a held-out
error rate can and cannot tell you. Results in R and Python need not match; that is expected.

### Data Description & Scientific Question

The data are Kepler Objects of Interest (KOIs) from the [NASA Exoplanet Archive cumulative
table](https://exoplanetarchive.ipac.caltech.edu). Each row is a transit-like signal around a Kepler star. The archive labels the object
as CONFIRMED (published planet), FALSE POSITIVE (the transit is not a planet), or CANDIDATE
(still under review).

Download kepler koi.csv and the codebook from Coursera. In the posted file, rows with label
CANDIDATE have already been dropped. The file has n = 7587 objects (2748 CONFIRMED, 4839
FALSE POSITIVE).

Response. The response is the two-level factor

`y = koi_disposition ∈ {CONFIRMED, FALSE POSITIVE}`.

Predictors. Use only the physical transit and stellar measurements in the posted file.
The column names are those in the codebook, including koi period, koi duration, koi depth,
koi impact, koi model snr, koi prad, koi teq, koi insol, koi steff, koi slogg, koi srad,
koi kepmag, and the other physical columns in the CSV. Units differ by orders of magnitude;
that is part of the problem.

Columns you may not use, even if you download a wider table yourself:

- any official false-positive flag (koi_fpflag_*);
- the Kepler pipeline score or disposition comment (koi_score, koi_pdisposition, koi_comment, and similar);
- names, identifiers, or sky coordinates used as features (kepoi_name is an ID only).

Those variables are how the archive recorded the label. Putting them in X turns the project into
a copy of the pipeline, not a classifier.

In [2]:
import pandas as pd

RANDOM_SEED = 598

df = pd.read_csv("kepler_koi.csv")

# only physical transit/stellar columns; excludes kepoi_name (ID) and koi_disposition (response)
predictor_cols = [
    "koi_period", "koi_duration", "koi_depth", "koi_impact", "koi_model_snr",
    "koi_prad", "koi_teq", "koi_insol", "koi_sma", "koi_incl", "koi_ror",
    "koi_steff", "koi_slogg", "koi_srad", "koi_smass", "koi_smet",
    "koi_kepmag", "koi_srho",
]

X = df[predictor_cols].copy()
y = df["koi_disposition"].copy()

print(X.shape)
print(y.value_counts())
X.head()

(7587, 18)
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
Name: count, dtype: int64


,koi_period,koi_duration,koi_depth,koi_impact,koi_model_snr,koi_prad,koi_teq,koi_insol,koi_sma,koi_incl,koi_ror,koi_steff,koi_slogg,koi_srad,koi_smass,koi_smet,koi_kepmag,koi_srho
0,2.470613,1.74319,14230.9,0.818,4304.3,13.04,1339.0,761.46,0.0354,84.41,0.123871,5820.0,4.457,0.964,0.971,-0.06,11.338,1.83538
1,2.204735,3.88864,6674.7,0.224,5945.9,16.10,2048.0,4148.92,0.0376,87.21,0.075522,6440.0,4.019,1.952,1.449,0.14,10.463,0.37936
2,4.887803,2.36255,4315.8,0.054,862.3,4.82,800.0,96.67,0.0530,89.82,0.057823,4778.0,4.591,0.763,0.830,0.32,9.174,3.69348
3,3.849372,2.63200,1298.6,0.915,132.7,12.94,2034.0,4055.29,0.0548,80.61,0.039601,6244.0,3.657,2.992,1.482,-0.16,11.432,0.22463
4,3.213669,3.98235,727.8,0.021,346.9,4.13,1515.0,1244.44,0.0440,89.81,0.024470,5795.0,4.101,1.545,1.097,0.18,12.211,0.46728


## Scientific Question

The scientific question is whether the transit geometry and the host-star properties, by themselves,
distinguish confirmed planets from false positives, and which classifier from the course
you would trust for that triage.

Design a fair comparison of classifiers for this two-class problem. The comparison has
to be something you could defend to an astronomer who will spend telescope time on the
objects you call planets.

Methods. Fit all of the following on the same training data, and evaluate them on the same
held-out test data:

(i) LDA;

(ii) logistic regression;

(iii) a linear SVM;

(iv) a kernel SVM (you choose the kernel);

(v) one additional method from the course that is not an SVM and not LDA/logistic. Reasonable
choices include a classification tree (or a forest), $L_1$ or $L_2$ penalized logistic regression,
or QDA. Explain why you picked it.

## Setup

Shared train/test split and missing-value imputation used by every method below.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.8,
    stratify=y, # stratify as required
    random_state=RANDOM_SEED,
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

# impute missing physical measurements using training-set medians only
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

### (i) LDA

In [4]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

lda = LinearDiscriminantAnalysis()
lda.fit(X_train_imp, y_train)

y_train_pred_lda = lda.predict(X_train_imp)
y_test_pred_lda = lda.predict(X_test_imp)

train_acc_lda = accuracy_score(y_train, y_train_pred_lda)
test_acc_lda = accuracy_score(y_test, y_test_pred_lda)

print(f"LDA train accuracy: {train_acc_lda:.4f}")
print(f"LDA test accuracy: {test_acc_lda:.4f}")
print("Test confusion matrix (rows=true, cols=predicted):")
print(pd.crosstab(y_test, y_test_pred_lda, rownames=["True"], colnames=["Predicted"]))

LDA train accuracy: 0.8192
LDA test accuracy: 0.8182
Test confusion matrix (rows=true, cols=predicted):
Predicted       CONFIRMED  FALSE POSITIVE
True                                     
CONFIRMED             428             122
FALSE POSITIVE        154             814


### (ii) Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# unpenalized logistic regression in the original units
# newton-cholesky handles the unscaled better than lbfgs
# https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
logreg = LogisticRegression(penalty=None, solver="newton-cholesky", max_iter=5000)
logreg.fit(X_train_imp, y_train)

y_train_pred_logreg = logreg.predict(X_train_imp)
y_test_pred_logreg = logreg.predict(X_test_imp)

train_acc_logreg = accuracy_score(y_train, y_train_pred_logreg)
test_acc_logreg = accuracy_score(y_test, y_test_pred_logreg)

print(f"Logistic regression train accuracy: {train_acc_logreg:.4f}")
print(f"Logistic regression test accuracy: {test_acc_logreg:.4f}")
print("Test confusion matrix (rows=true, cols=predicted):")
print(pd.crosstab(y_test, y_test_pred_logreg, rownames=["True"], colnames=["Predicted"]))

Logistic regression train accuracy: 0.8585
Logistic regression test accuracy: 0.8696
Test confusion matrix (rows=true, cols=predicted):
Predicted       CONFIRMED  FALSE POSITIVE
True                                     
CONFIRMED             486              64
FALSE POSITIVE        134             834


/Users/ryanlenea/Git/PSL/cs598_psl/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


### (iii) Linear SVM

In [10]:
#TODO Mark

### (iv) Kernel SVM

In [11]:
#TODO Liam

### (v) Additional Method

One additional method from the course that is not an SVM and not LDA/logistic. Reasonable
choices include a classification tree (or a forest), $L_1$ or $L_2$ penalized logistic regression,
or QDA. Explain why you picked it.

In [12]:
#TODO Anirudh

## Protocol

You choose the split and the tuning scheme, subject to these constraints:

- There must be a held-out test set that is not used to choose C, a kernel width, a tree depth, a penalty, or anything else.
- The split must be stratified by the two dispositions.
- Missing values: drop or impute using training-set information only; explain what you did.
- Scale features when the method uses distances or inner products, using training-set summaries only. LDA and unpenalized logistic regression in the original units are defensible; explain what you scaled and why.
- Tuning is done by cross-validation on the training set only. At minimum you must tune C for both SVMs and the kernel width if you use a radial kernel.
- The whole pipeline must be reproducible: report the seed, the split fractions, the CV folds, and the grids.

Default training-set frequencies are acceptable class priors. If you change priors or the relative
cost of the two errors, explain.

## What to Report

The report should contain enough for a reader to reconstruct what you did and to judge the
scientific claim. At a minimum:

(a) A short description of the protocol (split, missing values, scaling, kernel, grids). Do not paste pages of raw software output!!

#TODO Ryan

(b) A comparison table with training and test error (or accuracy) for every method. Confusion
matrices on the test set, with rows = true disposition and columns = predicted disposition,
labeled by name rather than 0/1. Report the two error types separately: confirmed planets
called false positives, and false positives called planets. Overall accuracy is not enough!
#TODO Mark

(c) At least one figure that is not a confusion matrix: period versus planet radius by class, a
CV curve in C, support-vector counts versus C, a calibration plot of logistic probabilities
versus SVM scores; you choose. The caption should state what the figure is evidence for.
#TODO Liam

(d) A written discussion, not a bullet paraphrase of the table. Address all of the following,
with reference to the numbers and figure(s) you produced:

- Which method would you hand an observer who has to decide which KOIs get followup spectra, and why? Test accuracy alone is not a complete answer. Which error is worse in this application, and did your ranking of methods change when you looked at the two error types separately? # TODO Anirudh
- Linear SVM versus logistic regression: do they disagree on many test objects? Relate this to hinge loss versus logistic loss (points beyond the margin; a probability model versus a score). TODO Anirudh
- Does the kernel SVM improve on the linear SVM enough to justify the extra tuning? Does a very large C look like overfit? TODO Ryan
- SVM scores are not b P(CONFIRMED | x). If the observer wanted a “needs more data” bin instead of a hard call, which of your methods is actually built for that? Say what you would do instead of treating sign(f(x)) as a posterior. TODO Mark
- One limitation of the study (missingness that is not at random, confirmed planets being easier targets, a single split, residual label leakage if you slipped a flag into X, ...) and how it should change how strongly you believe the ranking of methods. TODO LIam

## Grading

Roughly equal weight on:

1. a correct and reproducible pipeline, including the column restrictions above
2. a comparison that includes both SVMs and the earlier classifiers, with tuning done on training data only;
3. figures and tables that a reader can interpret without guessing, including both kinds of error;
4. a discussion that uses the language of the course rather than “model A had a slightly higher accuracy.” A summary of test accuracies with no argument is not a completed project.

Remark: A tenth of a percentage point in test accuracy is not a finding. Discuss the pattern
of errors and whether the method is doing something you would trust on the next KOI delivery.